In [1]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
from IPython.display import display, Markdown
import os
import sys
import asyncio
from openai import AsyncOpenAI
from agents import set_default_openai_client

if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

load_dotenv(override=True)

groq_client = AsyncOpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.getenv("GROQ_API_KEY")
)
set_default_openai_client(groq_client)
print("✅ Using Groq with llama-3.3-70b-versatile")

✅ Using Groq with llama-3.3-70b-versatile


In [ ]:
from accounts import Account
account = Account.get("ed")
account

In [3]:
account.buy_shares("AMZN", 3, "Because this bookstore website looks promising")

'Completed. Latest details:\n{"name": "ed", "balance": 9850.0, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 150.3, "timestamp": "2025-03-09 15:30:45", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [], "total_portfolio_value": 10300.0, "total_profit_loss": 450.0}'

In [4]:
account.report()

'{"name": "ed", "balance": 9850.0, "strategy": "", "holdings": {"AMZN": 3}, "transactions": [{"symbol": "AMZN", "quantity": 3, "price": 150.3, "timestamp": "2025-03-09 15:30:45", "rationale": "Because this bookstore website looks promising"}], "portfolio_value_time_series": [["2025-03-09 15:30:46", 10300.0]], "total_portfolio_value": 10300.0, "total_profit_loss": 450.0}'

In [5]:
account.list_transactions()

[{'symbol': 'AMZN',
  'quantity': 3,
  'price': 150.3,
  'timestamp': '2025-03-09 15:30:45',
  'rationale': 'Because this bookstore website looks promising'}]

In [6]:
params = {"command": sys.executable, "args": ["accounts_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='get_balance', description='Get the cash balance of the given account name.', input_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}),
 Tool(name='get_holdings', description='Get the holdings of the given account name.', input_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}),
 Tool(name='buy_shares', description='Buy shares of a stock.', input_schema={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'symbol': {'title': 'Symbol', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}, 'rationale': {'title': 'Rationale', 'type': 'string'}}, 'required': ['name', 'symbol', 'quantity', 'rationale'], 'title': 'buy_sharesArguments', 'type': 'object'}),
 Tool(name='sell_shares', description='Sell shares of a stock.', input_schema={'properties': {'name': {'title': 'Name',

In [7]:
instructions = "You are able to manage an account for a client, and answer questions about the account."
request = "My name is Ed and my account is under the name Ed. What's my balance and my holdings?"
model = "groq/llama-3.3-70b-versatile"

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="account_manager", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("account_manager"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Your current balance is $9,850.00 and your holdings consist of 3 shares of AMZN.

In [8]:
from accounts_client import get_accounts_tools_openai, read_accounts_resource, list_accounts_tools

mcp_tools = await list_accounts_tools()
print(mcp_tools)

[Tool(name='get_balance', description='Get the cash balance of the given account name.', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_balanceArguments', 'type': 'object'}), Tool(name='get_holdings', description='Get the holdings of the given account name.', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}}, 'required': ['name'], 'title': 'get_holdingsArguments', 'type': 'object'}), Tool(name='buy_shares', description='Buy shares of a stock.', inputSchema={'properties': {'name': {'title': 'Name', 'type': 'string'}, 'symbol': {'title': 'Symbol', 'type': 'string'}, 'quantity': {'title': 'Quantity', 'type': 'integer'}, 'rationale': {'title': 'Rationale', 'type': 'string'}}, 'required': ['name', 'symbol', 'quantity', 'rationale'], 'title': 'buy_sharesArguments', 'type': 'object'}), Tool(name='sell_shares', description='Sell shares of a stock.', inputSchema={'properties': {'name': {'title': 'Name', 'type'

In [9]:
context = await read_accounts_resource("ed")
print(context)

Account for ed:
Balance: $9,850.00
Holdings: 3 shares of AMZN
